# Notebook 2 — Evaluate and run Sen2-SSIR

Place exactly one Sentinel-2 L2A product under `Data/Testing_Scene`. This notebook first performs the simulated 20 m/40 m evaluation and then produces a real ten-band 10 m GeoTIFF.

In [ ]:
import tensorflow as tf

from utils import (
    EVALUATION_RESULTS_DIR, INFERENCE_RESULTS_DIR, PRETRAINED_WEIGHTS,
    TESTING_SCENE_DIR, InferenceConfig, ensure_project_directories,
    evaluate_scene, load_sen2_ssir_weights, require_single_scene,
    superresolve_scene_to_geotiff,
)

ensure_project_directories()
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## Configuration

Keep `ROI = None` for the whole scene inference.

In [ ]:
ROI = None              # or (x0, y0, x1, y1), in native 10 m pixels
WEIGHTS_PATH = PRETRAINED_WEIGHTS # Change the trained model's weight if needed
GEOTIFF_DTYPE = 'float32'       # 'uint16' (compact) or 'float32' (full precision)
GEOTIFF_COMPRESSION = 'LZW'   # 'ZSTD', 'DEFLATE', 'LZW', or 'NONE'
GEOTIFF_COMPRESSION_LEVEL = 9  # 1 (fast) to 9 (smaller)
inference_config = InferenceConfig(
    patch_size=128,
    border=4,
    batch_size=32,       # manuscript setting; lower if GPU/RAM is limited
    output_scale=10000.0,
    geotiff_dtype=GEOTIFF_DTYPE,
    geotiff_compression=GEOTIFF_COMPRESSION,
    geotiff_compression_level=GEOTIFF_COMPRESSION_LEVEL,
)

scene_path = require_single_scene(TESTING_SCENE_DIR)
print('Testing scene:', scene_path)
model = load_sen2_ssir_weights(WEIGHTS_PATH)
print(f'Sen2-SSIR parameters: {model.count_params():,}')

## Simulated evaluation

The native six 20 m bands are the reference. Their simulated 40 m versions and the four fine bands degraded to 20 m are passed to Sen2-SSIR. The JSON report contains RMSE,SRE, and ERGAS with `r=0.25`, both in aggregate and for every predicted band. Predictions and metrics are saved under `Results/Evaluation`.

In [ ]:
metrics, prediction_path, metrics_path = evaluate_scene(
    model,
    scene_path,
    EVALUATION_RESULTS_DIR,
    config=inference_config,
    roi=ROI,
    verbose=True,
)
print('Manuscript simulated-evaluation metrics:')
for metric_name, value in metrics['aggregate'].items():
    print(f'  {metric_name:12s}: {value:.6f}')
print('Per-band manuscript metrics:')
for band_name, band_metrics in metrics['per_band'].items():
    values = ', '.join(f'{name}={value:.6f}' for name, value in band_metrics.items())
    print(f'  {band_name:3s}: {values}')
print('Prediction:', prediction_path)
print('Metric report:', metrics_path)

## Real 10 m inference

The output is a compressed, georeferenced GeoTIFF in band order `B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12`. B2/B3/B4/B8 are native bands; the other six are Sen2-SSIR predictions. Compact `uint16` stores rounded Sentinel-style digital numbers and records a `0.0001` band scale. Full-precision `float32` stores surface reflectance directly.

In [ ]:
roi_suffix = '' if ROI is None else '_roi_' + '_'.join(str(value) for value in ROI)
export_suffix = f"{GEOTIFF_DTYPE.lower()}_{GEOTIFF_COMPRESSION.lower()}"
output_geotiff = INFERENCE_RESULTS_DIR / f"{metrics['scene_id']}{roi_suffix}_Sen2-SSIR_10m_10bands_{export_suffix}.tif"
output_geotiff = superresolve_scene_to_geotiff(
    model,
    scene_path,
    output_geotiff,
    config=inference_config,
    roi=ROI,
    verbose=True,
)
print('Ten-band output:', output_geotiff)